## Cell 1 — Imports

import pandas as pd
import numpy as np
import io
import os
import json
import ast
from pathlib import Path
from google.cloud import storage

## Cell 2 — Global params (your real values)

In [25]:
class Globals:
    gcp_project = 'dx-scin-public'
    gcs_bucket_name = 'dx-scin-public-data'
    cases_csv = 'dataset/scin_cases.csv'
    labels_csv = 'dataset/scin_labels.csv'
    gcs_images_dir = 'dataset/images/'

    image_path_columns = ['image_1_path', 'image_2_path', 'image_3_path']
    weighted_skin_condition_label = "weighted_skin_condition_label"
    skin_condition_label = "dermatologist_skin_condition_on_label_name"

    gcs_storage_client = None
    gcs_bucket = None
    cases_df = None
    cases_and_labels_df = None

In [26]:
print('GCS bucket name:', Globals.gcs_bucket_name)
print('cases_csv:', Globals.cases_csv)
print('labels_csv:', Globals.labels_csv)
print('weighted label col:', Globals.weighted_skin_condition_label)

GCS bucket name: dx-scin-public-data
cases_csv: dataset/scin_cases.csv
labels_csv: dataset/scin_labels.csv
weighted label col: weighted_skin_condition_label


## Connect and load metadata (anonymous client, since public bucket)

In [27]:
def initialize_df_with_metadata(bucket, csv_path):
    df = pd.read_csv(io.BytesIO(bucket.blob(csv_path).download_as_string()), dtype={'case_id': str})
    df['case_id'] = df['case_id'].astype(str)
    return df

def augment_metadata_with_labels(df, bucket, csv_path):
    labels_df = pd.read_csv(io.BytesIO(bucket.blob(csv_path).download_as_string()), dtype={'case_id': str})
    labels_df['case_id'] = labels_df['case_id'].astype(str)
    return pd.merge(df, labels_df, on='case_id')

Globals.gcs_storage_client = storage.Client.create_anonymous_client()
Globals.gcs_bucket = Globals.gcs_storage_client.bucket(Globals.gcs_bucket_name)
Globals.cases_df = initialize_df_with_metadata(Globals.gcs_bucket, Globals.cases_csv)
Globals.cases_and_labels_df = augment_metadata_with_labels(Globals.cases_df, Globals.gcs_bucket, Globals.labels_csv)
print(len(Globals.cases_and_labels_df))

5033


## Parse weighted labels

In [28]:
def safe_parse(x):
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError, TypeError):
        return {}

Globals.cases_and_labels_df['weighted_label_parsed'] = (
    Globals.cases_and_labels_df[Globals.weighted_skin_condition_label].apply(safe_parse)
)

## Cell 5 — Get primary label

In [32]:
def get_primary_label(d):
    if not d:
        return None
    return max(d, key=d.get)

Globals.cases_and_labels_df['primary_label'] = (
    Globals.cases_and_labels_df['weighted_label_parsed'].apply(get_primary_label)
)
print("Missing labels:", Globals.cases_and_labels_df['primary_label'].isna().sum())

Missing labels: 1972


## Cell 6 — Class selection

In [34]:
selected_classes = [
    'Eczema', 'Allergic Contact Dermatitis', 'Urticaria', 'Insect Bite',
    'Folliculitis', 'Psoriasis', 'Tinea', 'Impetigo'
]

df_labeled = Globals.cases_and_labels_df[
    Globals.cases_and_labels_df['primary_label'].notna()
].copy()

df_labeled['model_class'] = df_labeled['primary_label'].apply(
    lambda x: x if x in selected_classes else 'Other'
)

print(df_labeled['model_class'].value_counts())

model_class
Other                          1491
Eczema                          488
Allergic Contact Dermatitis     270
Urticaria                       214
Insect Bite                     185
Folliculitis                    142
Psoriasis                       109
Tinea                            93
Impetigo                         69
Name: count, dtype: int64


## Cell 7 — Save labeled dataframe

In [35]:
Path('data').mkdir(exist_ok=True)
df_labeled.to_csv('data/scin_labeled_cases.csv', index=False)
print(f"Saved {len(df_labeled)} labeled cases")

Saved 3061 labeled cases


## Cell 8 — Download setup

In [36]:
OUTPUT_DIR = Path('data/scin_images')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def download_image(bucket, blob_path, dest_path):
    if dest_path.exists() and dest_path.stat().st_size > 0:
        return True
    try:
        blob = bucket.blob(blob_path)
        blob.download_to_filename(str(dest_path))
        return True
    except Exception as e:
        print(f"FAILED: {blob_path} -> {e}")
        return False

for cls in df_labeled['model_class'].unique():
    (OUTPUT_DIR / cls).mkdir(exist_ok=True)

## Cell 9 — Full download (will skip already-downloaded files instantly)

In [37]:
CHECKPOINT_FILE = Path("data/download_progress.json")

success_count = 0
fail_count = 0
failed_cases = []

for i, (_, row) in enumerate(df_labeled.iterrows(), start=1):
    case_id = row["case_id"]
    cls = row["model_class"]
    blob_path = row["image_1_path"]
    ext = os.path.splitext(blob_path)[1]
    dest_path = OUTPUT_DIR / cls / f"{case_id}{ext}"

    if dest_path.exists() and dest_path.stat().st_size > 0:
        success_count += 1
        continue

    ok = download_image(Globals.gcs_bucket, blob_path, dest_path)
    if ok:
        success_count += 1
    else:
        fail_count += 1
        failed_cases.append(case_id)

    if i % 100 == 0:
        print(f"Progress: {i}/{len(df_labeled)} | success: {success_count} | failed: {fail_count}")

with open(CHECKPOINT_FILE, "w") as f:
    json.dump({"success_count": success_count, "fail_count": fail_count, "failed_cases": failed_cases}, f)

print(f"\n===== DONE =====\nSuccess: {success_count}\nFailed: {fail_count}")

FAILED: dataset/images/-2243186711511406658.png -> 404 GET https://storage.googleapis.com/download/storage/v1/b/dx-scin-public-data/o/dataset%2Fimages%2F-2243186711511406658.png?alt=media: No such object: dx-scin-public-data/dataset/images/-2243186711511406658.png: ('Request failed with status code', 404, 'Expected one of', <HTTPStatus.OK: 200>, <HTTPStatus.PARTIAL_CONTENT: 206>)

===== DONE =====
Success: 3060
Failed: 1
